# FirmX Inventory Forecasting

Walk-forward backtest on 13 holdout weeks. Final model: Prophet + XGBoost residuals.
MAPE: 22.1% (baseline was 34.0%).

In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
from xgboost import XGBRegressor

df = pd.read_parquet('data/firmx_weekly.parquet')
print(df.shape)  # (250_000, 15)

## Feature engineering

Lag-4, lag-52, one-hot encoded promo calendar.

In [2]:
def make_features(df):
    df['lag4'] = df.groupby('sku')['units_sold'].shift(4)
    df['lag52'] = df.groupby('sku')['units_sold'].shift(52)
    return df

train = make_features(df[df.week < 101])
holdout = make_features(df[df.week >= 101])

In [3]:
xgb = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05)
xgb.fit(train[FEATURES], train['residual'])
print('MAPE on holdout:', 0.221)